In [1]:

from IPython.display import display, Markdown
import pandas as pd

def section(title, text):
    display(Markdown(f"## {title}\n\n{text}"))

# Assignment title and objective
display(Markdown("""
# Assignment 7.1: Data Preprocessing

**Course:** AIT 620, Methods of Artificial Intelligence and Machine Learning

**Topic:** Module 7, Clustering

**Dataset:** United Nations Population Division, International Migration Flows to and from Selected Countries: The 2015 Revision (Canada by Citizenship)

## Objective

Restructure the original immigration worksheet into a country-indexed table containing Continent, Region, DevName, immigration counts for 1980–2013, and a Total column. The original workbook has descriptive title rows, so the actual header must be selected explicitly.
"""))

# 1. Import pandas and load the worksheet
section(
    "1. Import pandas and load the correct worksheet",
    "The workbook contains multiple worksheets. The country-level records appear in the Canada by Citizenship worksheet. The field names are located on Excel row 21, so header=20 is used in pandas. The dataset is loaded directly from the provided GitHub file link."
)

file_path = "https://github.com/dolehabdullah-boop/Assignment_7_1.ipynb/raw/refs/heads/main/Canada.xlsx"

raw_df = pd.read_excel(
    file_path,
    sheet_name="Canada by Citizenship",
    header=20
)

print("Original dataset shape:", raw_df.shape)
print("Original column names:", raw_df.columns.tolist())
display(raw_df.head())

# 2. Inspect missing data and select relevant columns
section(
    "2. Inspect missing data and select relevant columns",
    "The output only needs country names, three descriptive categories, and the 34 annual immigration columns. Removing the classification and numeric geographic codes avoids including non-immigration variables in the annual totals. The code checks that all required columns exist and reports missing values."
)

years = list(range(1980, 2014))

required_columns = [
    "OdName",
    "AreaName",
    "RegName",
    "DevName",
    *years
]

missing_columns = [
    column for column in required_columns
    if column not in raw_df.columns
]

if missing_columns:
    raise ValueError(
        f"Required columns are missing: {missing_columns}"
    )

print("Missing values in selected fields:")
missing_values = raw_df[required_columns].isna().sum()
print(missing_values.loc[missing_values > 0])

print("Number of annual columns:", len(years))

# 3. Rename fields and use country names as the index
section(
    "3. Rename fields and use country names as the index",
    "The assignment uses Country as the index name rather than a regular column. The original fields are renamed to Country, Continent, and Region where appropriate. The set_index() method makes Country the index, while DevName retains its original name."
)

immigration_df = (
    raw_df[required_columns]
    .rename(columns={
        "OdName": "Country",
        "AreaName": "Continent",
        "RegName": "Region"
    })
    .set_index("Country")
)

print("Index name:", immigration_df.index.name)
print(
    "Columns after renaming:",
    immigration_df.columns[:6].tolist()
)

display(immigration_df.head())

# 4. Ensure yearly immigration values are numeric
section(
    "4. Ensure the yearly immigration values are numeric",
    "Annual immigration columns must contain numeric values before totals can be calculated. Nonnumeric markers are converted to missing values and then replaced with zero. The code reports how many entries were handled."
)

original_year_values = immigration_df[years].copy()

converted_year_values = original_year_values.apply(
    pd.to_numeric,
    errors="coerce"
)

invalid_count = int(
    (
        converted_year_values.isna()
        & original_year_values.notna()
    ).sum().sum()
)

missing_count = int(
    converted_year_values.isna().sum().sum()
)

immigration_df[years] = (
    converted_year_values
    .fillna(0)
    .astype("int64")
)

print(
    "Nonnumeric year entries converted to missing:",
    invalid_count
)

print(
    "Total missing annual entries filled with zero:",
    missing_count
)

# 5. Calculate immigration totals
section(
    "5. Calculate immigration totals for 1980–2013",
    "Each country's Total is calculated by adding its 34 annual immigration counts from 1980 through 2013. The Total column is placed after the year columns, as shown in the assignment example."
)

immigration_df["Total"] = immigration_df[years].sum(axis=1)

print("Final DataFrame shape:", immigration_df.shape)
display(immigration_df.head())

# 6. Verify the output against the assignment example
section(
    "6. Verify the output against the assignment example",
    "The first five country totals shown in the assignment example are used as a reproducible check. The code also verifies the year range, column order, index name, and total number of columns."
)

expected_totals = {
    "Afghanistan": 58639,
    "Albania": 15699,
    "Algeria": 69439,
    "American Samoa": 6,
    "Andorra": 15
}

for country, expected in expected_totals.items():
    if country not in immigration_df.index:
        raise AssertionError(
            f"{country} is not present in the dataset."
        )

    actual = int(immigration_df.loc[country, "Total"])

    assert actual == expected, (
        f"{country}: expected {expected}, received {actual}"
    )

assert immigration_df.index.name == "Country"

assert immigration_df.columns.tolist() == [
    "Continent",
    "Region",
    "DevName",
    *years,
    "Total"
]

assert len(years) == 34
assert immigration_df.shape[1] == 38

print("PASS: All five country totals match the assignment example.")
print("PASS: Country index, column order, and 38-column layout are correct.")

display(
    immigration_df.loc[
        list(expected_totals),
        [
            "Continent",
            "Region",
            "DevName",
            1980,
            1981,
            2012,
            2013,
            "Total"
        ]
    ]
)

# 7. Final observations
section(
    "7. Final observations",
    "The original worksheet contains 197 rows, including entries named Unknown and Total. These entries are retained because the assignment focuses on restructuring the data rather than removing records. For a later clustering task, aggregate and non-country entries should be assessed before model training. The final table retains all 34 annual values and adds the calculated Total column."
)

print("Final DataFrame shape:", immigration_df.shape)
display(immigration_df.head())

# 8. Save the completed dataset
section(
    "8. Save the completed dataset",
    "The completed DataFrame is saved in Excel and CSV formats for future analysis."
)

csv_file = "Assignment_7_1_completed.csv"
excel_file = "Assignment_7_1_completed.xlsx"

immigration_df.to_csv(csv_file)
immigration_df.to_excel(excel_file)

print("Saved:", csv_file)
print("Saved:", excel_file)

# Reference
display(Markdown("""
## Reference

United Nations, Department of Economic and Social Affairs, Population Division. (2015). *International migration flows to and from selected countries: The 2015 revision* [Data set]. United Nations.
"""))

print("Assignment 7.1 preprocessing completed successfully.")



# Assignment 7.1: Data Preprocessing

**Course:** AIT 620, Methods of Artificial Intelligence and Machine Learning

**Topic:** Module 7, Clustering

**Dataset:** United Nations Population Division, International Migration Flows to and from Selected Countries: The 2015 Revision (Canada by Citizenship)

## Objective

Restructure the original immigration worksheet into a country-indexed table containing Continent, Region, DevName, immigration counts for 1980–2013, and a Total column. The original workbook has descriptive title rows, so the actual header must be selected explicitly.


## 1. Import pandas and load the correct worksheet

The workbook contains multiple worksheets. The country-level records appear in the Canada by Citizenship worksheet. The field names are located on Excel row 21, so header=20 is used in pandas. The dataset is loaded directly from the provided GitHub file link.

Original dataset shape: (197, 43)
Original column names: ['Type', 'Coverage', 'OdName', 'AREA', 'AreaName', 'REG', 'RegName', 'DEV', 'DevName', 1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013]


,Type,Coverage,OdName,AREA,AreaName,REG,RegName,DEV,DevName,1980,...,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013
0,Immigrants,Foreigners,Afghanistan,935,Asia,5501,Southern Asia,902,Developing regions,16,...,2978,3436,3009,2652,2111,1746,1758,2203,2635,2004
1,Immigrants,Foreigners,Albania,908,Europe,925,Southern Europe,901,Developed regions,1,...,1450,1223,856,702,560,716,561,539,620,603
2,Immigrants,Foreigners,Algeria,903,Africa,912,Northern Africa,902,Developing regions,80,...,3616,3626,4807,3623,4005,5393,4752,4325,3774,4331
3,Immigrants,Foreigners,American Samoa,909,Oceania,957,Polynesia,902,Developing regions,0,...,0,0,1,0,0,0,0,0,0,0
4,Immigrants,Foreigners,Andorra,908,Europe,925,Southern Europe,901,Developed regions,0,...,0,0,1,1,0,0,0,0,1,1


## 2. Inspect missing data and select relevant columns

The output only needs country names, three descriptive categories, and the 34 annual immigration columns. Removing the classification and numeric geographic codes avoids including non-immigration variables in the annual totals. The code checks that all required columns exist and reports missing values.

...

,Continent,Region,DevName,1980,1981,1982,1983,1984,1985,1986,...,2005,2006,2007,2008,2009,2010,2011,2012,2013,Total
Country,,,,,,,,,,,,,,,,,,,,,
Afghanistan,Asia,Southern Asia,Developing regions,16,39,39,47,71,340,496,...,3436,3009,2652,2111,1746,1758,2203,2635,2004,58639
Albania,Europe,Southern Europe,Developed regions,1,0,0,0,0,0,1,...,1223,856,702,560,716,561,539,620,603,15699
Algeria,Africa,Northern Africa,Developing regions,80,67,71,69,63,44,69,...,3626,4807,3623,4005,5393,4752,4325,3774,4331,69439
American Samoa,Oceania,Polynesia,Developing regions,0,1,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,6
Andorra,Europe,Southern Europe,Developed regions,0,0,0,0,0,0,2,...,0,1,1,0,0,0,0,1,1,15


## 8. Save the completed dataset

The completed DataFrame is saved in Excel and CSV formats for future analysis.

Saved: Assignment_7_1_completed.csv
Saved: Assignment_7_1_completed.xlsx



## Reference

United Nations, Department of Economic and Social Affairs, Population Division. (2015). *International migration flows to and from selected countries: The 2015 revision* [Data set]. United Nations.


Assignment 7.1 preprocessing completed successfully.
